# 🛒 Projet E-Commerce — Analyse Complète

**Données** : 2 000 commandes · 500 clients · 100 produits · 5 890 lignes de commande

| Section | Contenu |
|---|---|
| **1** | Exploration générale & schéma de modélisation |
| **2** | Analyse annulations & retours |
| **3** | Analyse RFM clients |
| **4** | Rentabilité réelle par catégorie |
| **5** | Performance des codes promo |
| **6** | Saisonnalité & tendances temporelles |
| **7** | Analyse géographique |
| **8** | Analyse des stocks |
| **9** | Modèles prédictifs (Churn · CA · Propension) |


## ⚙️ Configuration & Chargement des données


In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib
matplotlib.use('Agg')  # Pas de fenêtre popup — graphiques inline
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (14, 5)})

# ── Chemins (adapter si besoin) ───────────────────────────────────────
DATA_DIR   = Path('../data')
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Chargement ────────────────────────────────────────────────────────
commandes  = pd.read_csv(DATA_DIR / 'data_commandes.csv',        sep=';')
lignes     = pd.read_csv(DATA_DIR / 'data_lignes_commandes.csv', sep=';')
clients    = pd.read_csv(DATA_DIR / 'data_clients.csv',          sep=';')
produits   = pd.read_csv(DATA_DIR / 'data_produits.csv',         sep=';')
calendrier = pd.read_csv(DATA_DIR / 'dim_calendrier.csv',        sep=';')
with open(DATA_DIR / 'api_produits.json') as f:
    api_produits = json.load(f)

# ── Pré-traitements communs ───────────────────────────────────────────
commandes['date_commande']  = pd.to_datetime(commandes['date_commande'])
clients['date_inscription'] = pd.to_datetime(clients['date_inscription'], dayfirst=True)
calendrier['date_complete'] = pd.to_datetime(calendrier['date_complete'])
commandes['annee']          = commandes['date_commande'].dt.year
commandes['mois_num']       = commandes['date_commande'].dt.month
commandes['mois']           = commandes['date_commande'].dt.to_period('M')
commandes['mois_seq']       = (commandes['annee'] - 2023) * 12 + commandes['mois_num']
commandes['jour_semaine']   = commandes['date_commande'].dt.day_name()
commandes['heure']          = commandes['heure_commande'].str[:2].astype(int)
commandes['trimestre']      = commandes['date_commande'].dt.quarter
commandes['a_promo']        = commandes['code_promo'].notna()
produits['marge_brute']     = produits['prix_unitaire'] - produits['cout_achat']
produits['taux_marge']      = (produits['marge_brute'] / produits['prix_unitaire'] * 100).round(2)
df_clients = commandes.merge(clients[['client_id','segment','region','canal_acquisition']], on='client_id')

print('✓ Données chargées')
print(f'  Commandes  : {len(commandes):,}')
print(f'  Clients    : {len(clients):,}')
print(f'  Produits   : {len(produits):,}')
print(f'  Lignes cmd : {len(lignes):,}')

In [ ]:
# =============================================================================
# PROJET E-COMMERCE — ANALYSE COMPLÈTE
# Notebook 01 : Exploration générale + Schéma de modélisation + Annulations/Retours
# =============================================================================
# Auteur  : Analyse générée avec Claude (Anthropic)
# Données : data_commandes, data_clients, data_produits,
#           data_lignes_commandes, dim_calendrier, api_produits.json
# =============================================================================


# Style global

# -----------------------------------------------------------------------------
# 0. CHARGEMENT DES DONNÉES
# -----------------------------------------------------------------------------
# DATA_DIR et OUTPUT_DIR définis dans la cellule de configuration

commandes  = pd.read_csv(DATA_DIR / "data_commandes.csv",        sep=";")
lignes     = pd.read_csv(DATA_DIR / "data_lignes_commandes.csv", sep=";")
clients    = pd.read_csv(DATA_DIR / "data_clients.csv",          sep=";")
produits   = pd.read_csv(DATA_DIR / "data_produits.csv",         sep=";")
calendrier = pd.read_csv(DATA_DIR / "dim_calendrier.csv",        sep=";")

with open(DATA_DIR / "api_produits.json") as f:
    api_produits = json.load(f)

# Conversion des types
commandes["date_commande"]    = pd.to_datetime(commandes["date_commande"])
clients["date_inscription"]   = pd.to_datetime(clients["date_inscription"], dayfirst=True)
commandes["heure"]            = commandes["heure_commande"].str[:2].astype(int)
commandes["annee"]            = commandes["date_commande"].dt.year
commandes["mois_num"]         = commandes["date_commande"].dt.month
commandes["mois"]             = commandes["date_commande"].dt.to_period("M")
commandes["a_promo"]          = commandes["code_promo"].notna()

# Marges produits
produits["marge_brute"]  = produits["prix_unitaire"] - produits["cout_achat"]
produits["taux_marge"]   = (produits["marge_brute"] / produits["prix_unitaire"] * 100).round(2)

---
## Section 1 — EXPLORATION GÉNÉRALE


In [ ]:
print("=" * 65)
print("SECTION 1 — EXPLORATION GÉNÉRALE DES DONNÉES")
print("=" * 65)

# ── 1.1 Dimensions des tables ──────────────────────────────────────────────
print("\n[1.1] Dimensions des tables")
tables = {
    "data_commandes":         commandes,
    "data_lignes_commandes":  lignes,
    "data_clients":           clients,
    "data_produits":          produits,
    "dim_calendrier":         calendrier,
}
for name, df in tables.items():
    print(f"  {name:<30} {df.shape[0]:>6} lignes  ×  {df.shape[1]} colonnes")

# ── 1.2 Valeurs manquantes ─────────────────────────────────────────────────
print("\n[1.2] Valeurs manquantes")
for name, df in tables.items():
    nulls = df.isnull().sum()
    if nulls.sum() > 0:
        print(f"  {name}")
        print(nulls[nulls > 0].to_string())
    else:
        print(f"  {name} — aucune valeur manquante")

# Note : code_promo est vide pour ~51% des commandes → normal (pas de promo)

# ── 1.3 Distribution des statuts ──────────────────────────────────────────
print("\n[1.3] Distribution des statuts de commandes")
statuts = commandes["statut"].value_counts()
print(statuts)
print(f"\n  → Taux de commandes non livrées : "
      f"{(1 - statuts.get('Livrée',0) / len(commandes)) * 100:.1f}%")

# ── 1.4 Modes de paiement & transporteurs ─────────────────────────────────
print("\n[1.4] Modes de paiement")
print(commandes["mode_paiement"].value_counts())
print("\n[1.4] Transporteurs")
print(commandes["transporteur"].value_counts())

# ── 1.5 Statistiques des montants ─────────────────────────────────────────
print("\n[1.5] Statistiques des montants (commandes)")
print(commandes[["montant_ht", "montant_ttc", "frais_port"]].describe().round(2))

# ── 1.6 Segments & canaux clients ─────────────────────────────────────────
print("\n[1.6] Segments clients")
print(clients["segment"].value_counts())
print("\n[1.6] Canaux d'acquisition")
print(clients["canal_acquisition"].value_counts())
print("\n[1.6] Régions")
print(clients["region"].value_counts())

# ── 1.7 Catalogue produits ─────────────────────────────────────────────────
print("\n[1.7] Catégories produits")
print(produits["categorie"].value_counts())
print("\n[1.7] Statistiques prix / coût / stock")
print(produits[["prix_unitaire", "cout_achat", "stock_actuel", "note_moyenne"]].describe().round(2))

# ── 1.8 Produits en alerte stock ──────────────────────────────────────────
alerte_stock = produits[produits["stock_actuel"] <= produits["stock_min"]]
print(f"\n[1.8] Produits en alerte stock ({len(alerte_stock)} produits)")
print(alerte_stock[["nom_produit", "categorie", "stock_actuel", "stock_min"]])

# ── 1.9 CA global ─────────────────────────────────────────────────────────
ca_total  = commandes["montant_ttc"].sum()
ca_livre  = commandes[commandes["statut"] == "Livrée"]["montant_ttc"].sum()
print(f"\n[1.9] CA brut total       : {ca_total:>12,.0f} €")
print(f"      CA livré effectif   : {ca_livre:>12,.0f} €")
print(f"      Panier moyen global : {commandes['montant_ttc'].mean():>12,.0f} €")

# ── 1.10 Fréquence d'achat par client ─────────────────────────────────────
freq = commandes.groupby("client_id").size()
print(f"\n[1.10] Fréquence d'achat par client")
print(freq.describe().round(2))
print("\n  Distribution :")
print(freq.value_counts().sort_index())

# ── 1.11 Marges par catégorie ─────────────────────────────────────────────
print("\n[1.11] Marges par catégorie produit")
print(produits.groupby("categorie")[["prix_unitaire", "cout_achat", "taux_marge"]].mean().round(2))

# ── 1.12 CA par année ─────────────────────────────────────────────────────
print("\n[1.12] CA par année")
print(commandes.groupby("annee")["montant_ttc"].agg(["sum", "count", "mean"]).round(0))

---
## Section 2 — SCHÉMA DE MODÉLISATION (Star Schema)


In [ ]:
# Le modèle en étoile est le standard BI :
#   - 2 tables de FAITS   : FACT_COMMANDES, FACT_LIGNES_COMMANDES
#   - 3 tables de DIMENSION : DIM_CLIENTS, DIM_PRODUITS, DIM_CALENDRIER
#
# Relations :
#   DIM_CLIENTS      --[client_id]-->   FACT_COMMANDES
#   DIM_CALENDRIER   --[date]--->       FACT_COMMANDES
#   FACT_COMMANDES   --[commande_id]--> FACT_LIGNES_COMMANDES
#   DIM_PRODUITS     --[produit_id]-->  FACT_LIGNES_COMMANDES
#
# Ce schéma permet de filtrer/grouper les faits (ventes, montants)
# selon n'importe quelle dimension (qui, quand, quoi, où).

print("\n" + "=" * 65)
print("SECTION 2 — SCHÉMA DE MODÉLISATION (Star Schema)")
print("=" * 65)
print("""
  DIM_CLIENTS ──[client_id]──────────────────► FACT_COMMANDES
                                                    │
  DIM_CALENDRIER ──[date_commande]───────────────── │
                                                    │ [commande_id]
                                               FACT_LIGNES_COMMANDES
                                                    │
  DIM_PRODUITS ──[produit_id]─────────────────────► │
""")

# Vérification de la cohérence des clés étrangères
print("[Vérification des jointures]")
# Clients dans commandes
clients_inconnus = commandes[~commandes["client_id"].isin(clients["client_id"])]["client_id"].nunique()
print(f"  Commandes avec client_id inconnu : {clients_inconnus}")

# Produits dans lignes
produits_inconnus = lignes[~lignes["produit_id"].isin(produits["produit_id"])]["produit_id"].nunique()
print(f"  Lignes avec produit_id inconnu   : {produits_inconnus}")

# Commandes dans lignes
cmd_sans_lignes = commandes[~commandes["commande_id"].isin(lignes["commande_id"])]["commande_id"].nunique()
print(f"  Commandes sans lignes associées  : {cmd_sans_lignes}")

---
## Section 3 — ANALYSE ANNULATIONS & RETOURS


In [ ]:
print("\n" + "=" * 65)
print("SECTION 3 — ANALYSE ANNULATIONS & RETOURS")
print("=" * 65)

# Jointure commandes + clients
df = commandes.merge(clients[["client_id", "segment", "region", "canal_acquisition"]], on="client_id")

# ── 3.1 Vue globale ────────────────────────────────────────────────────────
print("\n[3.1] Vue globale par statut")
statuts_detail = commandes.groupby("statut").agg(
    nb        = ("commande_id", "count"),
    ca        = ("montant_ttc", "sum"),
    panier_moy= ("montant_ttc", "mean")
).assign(
    pct_nb = lambda x: (x["nb"] / len(commandes) * 100).round(1),
    pct_ca = lambda x: (x["ca"] / commandes["montant_ttc"].sum() * 100).round(1)
)
print(statuts_detail.round(0))

# CA perdu
ca_annule   = commandes[commandes["statut"] == "Annulée"]["montant_ttc"].sum()
ca_retourne = commandes[commandes["statut"] == "Retournée"]["montant_ttc"].sum()
ca_perdu    = ca_annule + ca_retourne
print(f"\n  ► CA annulé   : {ca_annule:>10,.0f} € ({ca_annule/ca_total*100:.1f}%)")
print(f"  ► CA retourné : {ca_retourne:>10,.0f} € ({ca_retourne/ca_total*100:.1f}%)")
print(f"  ► CA PERDU    : {ca_perdu:>10,.0f} € ({ca_perdu/ca_total*100:.1f}%)")

# ── 3.2 Taux par segment client ───────────────────────────────────────────
print("\n[3.2] Taux d'annulation & retour par segment client")
seg_stats = df.groupby("segment").apply(lambda x: pd.Series({
    "total"     : len(x),
    "taux_annul": round((x["statut"] == "Annulée").sum()   / len(x) * 100, 1),
    "taux_retour": round((x["statut"] == "Retournée").sum() / len(x) * 100, 1),
    "taux_total": round(x["statut"].isin(["Annulée","Retournée"]).sum() / len(x) * 100, 1),
    "ca_perdu"  : x[x["statut"].isin(["Annulée","Retournée"])]["montant_ttc"].sum()
})).sort_values("taux_total", ascending=False)
print(seg_stats)
# INSIGHT : Premium et Standard ont les taux les plus élevés (22-23%)
# alors qu'on attendrait l'inverse pour des clients "fidélisés"

# ── 3.3 Par mode de paiement ──────────────────────────────────────────────
print("\n[3.3] Taux par mode de paiement")
pmt_stats = df.groupby("mode_paiement").apply(lambda x: pd.Series({
    "total"      : len(x),
    "taux_annul" : round((x["statut"] == "Annulée").sum()   / len(x) * 100, 1),
    "taux_retour": round((x["statut"] == "Retournée").sum() / len(x) * 100, 1),
    "taux_total" : round(x["statut"].isin(["Annulée","Retournée"]).sum() / len(x) * 100, 1),
})).sort_values("taux_total", ascending=False)
print(pmt_stats)
# INSIGHT : PayPal = 16.2% vs Apple Pay = 22.0%
# Les paiements "frictionless" (Apple Pay, Klarna) favorisent l'achat impulsif → plus d'annulations

# ── 3.4 Par transporteur ──────────────────────────────────────────────────
print("\n[3.4] Taux par transporteur")
transp_stats = df.groupby("transporteur").apply(lambda x: pd.Series({
    "total"      : len(x),
    "taux_annul" : round((x["statut"] == "Annulée").sum()   / len(x) * 100, 1),
    "taux_retour": round((x["statut"] == "Retournée").sum() / len(x) * 100, 1),
    "taux_total" : round(x["statut"].isin(["Annulée","Retournée"]).sum() / len(x) * 100, 1),
})).sort_values("taux_total", ascending=False)
print(transp_stats)
# INSIGHT : DPD = 17.4% vs Colissimo = 22.3% → orientation DPD conseillée

# ── 3.5 Par catégorie produit ─────────────────────────────────────────────
print("\n[3.5] Taux par catégorie produit")
df_full = lignes.merge(produits[["produit_id", "categorie"]], on="produit_id")
df_full = df_full.merge(commandes[["commande_id", "statut"]], on="commande_id")
cmd_cat = df_full.drop_duplicates("commande_id")
cat_stats = cmd_cat.groupby("categorie").apply(lambda x: pd.Series({
    "nb_cmd"     : len(x),
    "taux_annul" : round((x["statut"] == "Annulée").sum()   / len(x) * 100, 1),
    "taux_retour": round((x["statut"] == "Retournée").sum() / len(x) * 100, 1),
    "taux_total" : round(x["statut"].isin(["Annulée","Retournée"]).sum() / len(x) * 100, 1),
})).sort_values("taux_total", ascending=False)
print(cat_stats)
# INSIGHT : Maison = 21.9% (produits encombrants, difficiles à retourner mais aussi
#           à évaluer en ligne → problème de représentation produit)

# ── 3.6 Par région ────────────────────────────────────────────────────────
print("\n[3.6] Taux par région")
reg_stats = df.groupby("region").apply(lambda x: pd.Series({
    "total"      : len(x),
    "taux_annul" : round((x["statut"] == "Annulée").sum()   / len(x) * 100, 1),
    "taux_retour": round((x["statut"] == "Retournée").sum() / len(x) * 100, 1),
    "taux_total" : round(x["statut"].isin(["Annulée","Retournée"]).sum() / len(x) * 100, 1),
    "ca_perdu"   : x[x["statut"].isin(["Annulée","Retournée"])]["montant_ttc"].sum()
})).sort_values("taux_total", ascending=False)
print(reg_stats.round(1))
# INSIGHT : Île-de-France = 25% — marché stratégique sous-performant

# ── 3.7 Impact codes promo ────────────────────────────────────────────────
print("\n[3.7] Impact des codes promo sur les annulations/retours")
promo_stats = commandes.groupby("a_promo").apply(lambda x: pd.Series({
    "total"      : len(x),
    "taux_annul" : round((x["statut"] == "Annulée").sum()   / len(x) * 100, 1),
    "taux_retour": round((x["statut"] == "Retournée").sum() / len(x) * 100, 1),
    "taux_total" : round(x["statut"].isin(["Annulée","Retournée"]).sum() / len(x) * 100, 1),
    "panier_moy" : x["montant_ttc"].mean().round(0)
}))
promo_stats.index = ["Sans promo", "Avec promo"]
print(promo_stats)
# INSIGHT CRITIQUE : +2.4 pts d'annulation avec promo (12% vs 8.8%)
# Les promos génèrent des achats impulsifs non aboutis → coût double (remise + non-vente)

# ── 3.8 Évolution mensuelle ───────────────────────────────────────────────
print("\n[3.8] Évolution mensuelle du taux de friction")
monthly = commandes.groupby("mois").apply(lambda x: pd.Series({
    "total"  : len(x),
    "annul"  : (x["statut"] == "Annulée").sum(),
    "retour" : (x["statut"] == "Retournée").sum(),
    "taux"   : round(x["statut"].isin(["Annulée","Retournée"]).sum() / len(x) * 100, 1)
}))
print(monthly)
# INSIGHT : Mai 2023 = 30.1% et Mai 2024 = 23.6% — deux pics consécutifs
# Probable corrélation avec les soldes / campagnes promotionnelles de mai

# ── 3.9 Panier moyen par statut ───────────────────────────────────────────
print("\n[3.9] Panier moyen & médian par statut")
print(commandes.groupby("statut")["montant_ttc"].agg(["mean", "median", "min", "max"]).round(0))
# Les commandes Retournées ont un panier médian plus élevé (1169 vs 979 pour Livrées)
# → les gros achats sont plus souvent retournés (produits chers, déception plus forte)

# ── 3.10 Top produits dans les commandes retournées ──────────────────────
print("\n[3.10] Top produits dans les commandes retournées")
cmd_retournees = commandes[commandes["statut"] == "Retournée"]["commande_id"]
lignes_retour  = lignes[lignes["commande_id"].isin(cmd_retournees)]
top_retour = (
    lignes_retour
    .merge(produits[["produit_id", "nom_produit", "categorie"]], on="produit_id")
    .groupby(["nom_produit", "categorie"])
    .agg(nb_fois=("ligne_id", "count"), ca_retourne=("montant_ligne", "sum"))
    .sort_values("nb_fois", ascending=False)
    .head(10)
)
print(top_retour)

---
## Section 4 — VISUALISATIONS


In [ ]:
print("\n[Section 4] Génération des visualisations...")
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Analyse Annulations & Retours — E-Commerce", fontsize=14, fontweight="bold")

# -- 4.1 Répartition CA par statut
ca_statut = commandes.groupby("statut")["montant_ttc"].sum() / 1e6
colors = {"Livrée": "#2ECC71", "En cours": "#3498DB", "Expédiée": "#85C1E9",
          "Annulée": "#E74C3C", "Retournée": "#F39C12"}
ax = axes[0, 0]
ca_statut.plot(kind="bar", ax=ax, color=[colors.get(s, "gray") for s in ca_statut.index])
ax.set_title("CA brut par statut (M€)")
ax.set_xlabel("")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f M€"))
ax.tick_params(axis="x", rotation=30)

# -- 4.2 Taux friction par segment
ax = axes[0, 1]
seg_plot = seg_stats[["taux_annul", "taux_retour"]].sort_values("taux_annul", ascending=True)
seg_plot.plot(kind="barh", ax=ax, color=["#E74C3C", "#F39C12"], stacked=True)
ax.set_title("Taux annul + retour par segment (%)")
ax.set_xlabel("Taux (%)")
ax.legend(["Annulation", "Retour"])

# -- 4.3 Taux friction par transporteur
ax = axes[0, 2]
transp_plot = transp_stats[["taux_annul", "taux_retour"]].sort_values("taux_annul", ascending=True)
transp_plot.plot(kind="barh", ax=ax, color=["#E74C3C", "#F39C12"], stacked=True)
ax.set_title("Taux annul + retour par transporteur (%)")
ax.set_xlabel("Taux (%)")
ax.legend(["Annulation", "Retour"])

# -- 4.4 Évolution mensuelle
ax = axes[1, 0]
monthly["taux"].plot(ax=ax, marker="o", color="#2980B9", linewidth=2)
ax.axhline(y=monthly["taux"].mean(), color="red", linestyle="--", linewidth=1, label="Moyenne")
ax.fill_between(range(len(monthly)), monthly["taux"].values,
                monthly["taux"].mean(), alpha=0.1, color="red")
ax.set_title("Évolution mensuelle du taux de friction (%)")
ax.set_xlabel("Mois")
ax.set_ylabel("Taux (%)")
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels([str(m) for m in monthly.index], rotation=45, fontsize=7)
ax.legend()

# -- 4.5 Promo vs sans promo
ax = axes[1, 1]
promo_plot = promo_stats[["taux_annul", "taux_retour"]]
promo_plot.plot(kind="bar", ax=ax, color=["#E74C3C", "#F39C12"])
ax.set_title("Impact codes promo sur friction (%)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
ax.legend(["Annulation", "Retour"])

# -- 4.6 Top produits retournés
ax = axes[1, 2]
top_retour.head(8)["nb_fois"].sort_values().plot(kind="barh", ax=ax, color="#E74C3C")
ax.set_title("Top produits dans commandes retournées")
ax.set_xlabel("Nb d'occurrences")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_annulations_retours.png", dpi=130, bbox_inches="tight")
plt.show()
print("\n  ✓ Graphique sauvegardé : outputs/01_annulations_retours.png")

---
## Section 5 — RÉSUMÉ DES INSIGHTS


In [ ]:
print("\n" + "=" * 65)
print("RÉSUMÉ DES INSIGHTS — ANNULATIONS & RETOURS")
print("=" * 65)

insights = [
    ("CA perdu total",
     f"{ca_perdu:,.0f} € soit {ca_perdu/ca_total*100:.1f}% du CA brut"),
    ("Segment le + problématique",
     "Premium (22.2%) et Standard (23.2%) — paradoxal pour des clients 'fidèles'"),
    ("Meilleur mode de paiement",
     "PayPal (16.2%) vs Apple Pay (22.0%) — 6 pts d'écart"),
    ("Meilleur transporteur",
     "DPD (17.4%) vs Colissimo (22.3%) — action rapide possible"),
    ("Impact promos",
     "Taux annulation +2.4 pts avec promo (12% vs 8.8%) — double coût"),
    ("Mois à risque",
     "Mai 2023 (30.1%) et Mai 2024 (23.6%) — effet soldes récurrent"),
    ("Région la + problématique",
     "Île-de-France (25.0%) — sous-performance sur marché stratégique"),
    ("Catégorie la + risquée",
     "Maison (21.9%) — produits encombrants difficiles à évaluer en ligne"),
]

for titre, detail in insights:
    print(f"\n  ► {titre}")
    print(f"    {detail}")

print("\n" + "=" * 65)
print("FIN DU NOTEBOOK 01")
print("=" * 65)

In [ ]:
# =============================================================================
# PROJET E-COMMERCE — ANALYSE COMPLÈTE
# Notebook 02 : Analyse RFM + Rentabilité par catégorie + Codes promo
# =============================================================================



# DATA_DIR et OUTPUT_DIR définis dans la cellule de configuration

commandes  = pd.read_csv(DATA_DIR / "data_commandes.csv",        sep=";")
lignes     = pd.read_csv(DATA_DIR / "data_lignes_commandes.csv", sep=";")
clients    = pd.read_csv(DATA_DIR / "data_clients.csv",          sep=";")
produits   = pd.read_csv(DATA_DIR / "data_produits.csv",         sep=";")

commandes["date_commande"] = pd.to_datetime(commandes["date_commande"])
commandes["a_promo"]       = commandes["code_promo"].notna()
produits["marge_brute"]    = produits["prix_unitaire"] - produits["cout_achat"]
produits["taux_marge"]     = (produits["marge_brute"] / produits["prix_unitaire"] * 100).round(2)

---
## Section 6 — ANALYSE RFM (Récence · Fréquence · Valeur)


In [ ]:
# Principe : chaque client reçoit un score de 1 à 5 sur 3 dimensions.
#   R (Récence)   : depuis combien de jours a-t-il passé sa dernière commande ?
#                   Score 5 = client très récent, Score 1 = client très ancien
#   F (Fréquence) : combien de commandes a-t-il passées ?
#                   Score 5 = très fréquent, Score 1 = rare
#   V (Valeur)    : quel est le CA total généré par ce client ?
#                   Score 5 = très haut CA, Score 1 = très faible CA
#
# On ne garde que les commandes "abouties" (livrées, en cours, expédiées)
# car les annulées/retournées ne génèrent pas de valeur réelle.

print("=" * 65)
print("SECTION 1 — ANALYSE RFM")
print("=" * 65)

# Date de référence = date de la commande la plus récente dans le dataset
date_ref = commandes["date_commande"].max()
print(f"\nDate de référence : {date_ref.date()}")

# Filtrer les commandes valides (exclure annulées et retournées)
cmd_valides = commandes[~commandes["statut"].isin(["Annulée", "Retournée"])]
print(f"Commandes valides : {len(cmd_valides)} / {len(commandes)} total")

# ── 1.1 Calcul des métriques R, F, V ──────────────────────────────────────
rfm = cmd_valides.groupby("client_id").agg(
    recence   = ("date_commande", lambda x: (date_ref - x.max()).days),
    frequence = ("commande_id",   "count"),
    valeur    = ("montant_ttc",   "sum")
).reset_index()

print(f"\nClients actifs analysés : {len(rfm)}")
print("\n[Statistiques RFM]")
print(rfm[["recence", "frequence", "valeur"]].describe().round(1))

# ── 1.2 Scoring par quintiles ─────────────────────────────────────────────
# On divise chaque métrique en 5 groupes de taille égale (quintiles).
# Pour la Récence : un client récent (faible nombre de jours) = score élevé
# → on inverse l'ordre avec labels=[5,4,3,2,1]
rfm["score_R"] = pd.qcut(rfm["recence"], 5, labels=[5,4,3,2,1]).astype(int)
rfm["score_F"] = pd.qcut(rfm["frequence"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["score_V"] = pd.qcut(rfm["valeur"].rank(method="first"),    5, labels=[1,2,3,4,5]).astype(int)
rfm["score_RFM"] = rfm["score_R"] + rfm["score_F"] + rfm["score_V"]

print(f"\nScore RFM min : {rfm['score_RFM'].min()} — max : {rfm['score_RFM'].max()}")
print(rfm["score_RFM"].value_counts().sort_index())

# ── 1.3 Segmentation métier ───────────────────────────────────────────────
# On traduit les scores en segments actionnables pour le marketing
def segment_rfm(row):
    r, f, v = row["score_R"], row["score_F"], row["score_V"]
    if r >= 4 and f >= 4 and v >= 4:
        return "Champions"               # récents, fréquents, gros CA → chouchoux
    elif r >= 3 and f >= 3 and v >= 4:
        return "Clients fidèles"         # bons clients mais moins récents
    elif r >= 4 and f <= 2:
        return "Nouveaux clients"        # récents mais peu d'achats → à développer
    elif r >= 3 and f >= 3 and v <= 3:
        return "En développement"        # engagés mais CA modeste → upsell
    elif r <= 2 and f >= 3 and v >= 3:
        return "A risque de churn"       # bons clients qui s'éloignent → urgence
    elif r <= 2 and f >= 4 and v >= 4:
        return "Clients perdus (VIP)"    # ex-champions disparus → réactivation max
    elif r <= 2 and f <= 2:
        return "Inactifs"                # peu fréquents et lointains → abandon
    else:
        return "Potentiel moyen"

rfm["segment_rfm"] = rfm.apply(segment_rfm, axis=1)

# ── 1.4 Analyse par segment ───────────────────────────────────────────────
print("\n[1.4] Distribution des segments RFM")
seg_rfm = rfm.groupby("segment_rfm").agg(
    nb_clients   = ("client_id",   "count"),
    recence_moy  = ("recence",     "mean"),
    freq_moy     = ("frequence",   "mean"),
    valeur_moy   = ("valeur",      "mean"),
    valeur_total = ("valeur",      "sum"),
    score_moyen  = ("score_RFM",   "mean")
).sort_values("valeur_total", ascending=False)
print(seg_rfm.round(0))

# ── 1.5 Top Champions ─────────────────────────────────────────────────────
print("\n[1.5] Top 15 Champions")
champions = (
    rfm[rfm["segment_rfm"] == "Champions"]
    .merge(clients[["client_id","prenom","nom","segment","region"]], on="client_id")
    .sort_values("valeur", ascending=False)
    .head(15)
)
print(champions[["prenom","nom","segment","region","recence","frequence","valeur","score_RFM"]])

# ── 1.6 Clients à risque de churn ─────────────────────────────────────────
print("\n[1.6] Clients à risque de churn")
churn = (
    rfm[rfm["segment_rfm"] == "A risque de churn"]
    .merge(clients[["client_id","prenom","nom","segment","region"]], on="client_id")
    .sort_values("valeur", ascending=False)
)
print(f"Nombre : {len(churn)} clients")
print(f"CA historique à récupérer : {churn['valeur'].sum():,.0f} €")
print(churn.head(10)[["prenom","nom","segment","recence","frequence","valeur"]])

# ── 1.7 Visualisation RFM ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Analyse RFM — Distribution des clients", fontsize=13, fontweight="bold")

seg_colors = {
    "Champions":            "#2ECC71",
    "Clients fidèles":      "#7F77DD",
    "A risque de churn":    "#F39C12",
    "En développement":     "#1ABC9C",
    "Nouveaux clients":     "#3498DB",
    "Inactifs":             "#E74C3C",
    "Potentiel moyen":      "#95A5A6",
}

# Répartition clients par segment
ax = axes[0]
data_pie = seg_rfm["nb_clients"]
colors = [seg_colors.get(s, "#ccc") for s in data_pie.index]
ax.pie(data_pie, labels=data_pie.index, colors=colors, autopct="%1.0f%%",
       startangle=140, textprops={"fontsize": 8})
ax.set_title("Répartition des clients")

# CA par segment
ax = axes[1]
data_ca = seg_rfm["valeur_total"].sort_values(ascending=True)
bars = ax.barh(data_ca.index, data_ca.values / 1e3,
               color=[seg_colors.get(s, "#ccc") for s in data_ca.index])
ax.set_title("CA total par segment (k€)")
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%dk€"))
ax.set_xlabel("CA (k€)")

# Scatter R vs V coloré par segment
ax = axes[2]
for seg, grp in rfm.groupby("segment_rfm"):
    ax.scatter(grp["recence"], grp["valeur"] / 1e3, s=20,
               c=seg_colors.get(seg, "#ccc"), label=seg, alpha=0.7)
ax.set_xlabel("Récence (jours depuis dernière commande)")
ax.set_ylabel("Valeur totale (k€)")
ax.set_title("Récence vs Valeur par segment")
ax.legend(fontsize=7, loc="upper right")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_rfm_analyse.png", dpi=130, bbox_inches="tight")
plt.show()
print("\n  ✓ Graphique sauvegardé : outputs/02_rfm_analyse.png")

---
## Section 7 — RENTABILITE REELLE PAR CATEGORIE


In [ ]:
# On calcule la marge brute réelle en croisant :
#   - les lignes de commandes (montant, remise)
#   - le taux de marge produit (prix - coût d'achat)
#   - le statut de la commande (on sépare livrées vs perdues)
#
# Cela donne la "marge nette" = marge sur livrées - marge sur perdues

print("\n" + "=" * 65)
print("SECTION 2 — RENTABILITE REELLE PAR CATEGORIE")
print("=" * 65)

# Jointure lignes + produits (on renomme pour éviter conflit de colonne)
produits2 = produits[["produit_id","nom_produit","categorie","sous_categorie",
                       "cout_achat","taux_marge"]].rename(columns={"taux_marge":"taux_marge_prod"})
df_full = (
    lignes
    .merge(produits2, on="produit_id")
    .merge(commandes[["commande_id","statut","montant_ttc","a_promo"]], on="commande_id")
)
# Marge réelle par ligne = montant_ligne × taux_marge du produit
df_full["marge_ligne"]    = df_full["montant_ligne"] * df_full["taux_marge_prod"] / 100
# Remise accordée = (prix × qté) - montant_ligne (après remise)
df_full["remise_valeur"]  = (df_full["prix_unitaire"] * df_full["quantite"]) - df_full["montant_ligne"]

df_livres = df_full[df_full["statut"] == "Livrée"]
df_perdus = df_full[df_full["statut"].isin(["Annulée","Retournée"])]

# ── 2.1 Rentabilité par catégorie (livrées uniquement) ────────────────────
print("\n[2.1] Rentabilité brute par catégorie (commandes livrées)")
renta = df_livres.groupby("categorie").agg(
    ca        = ("montant_ligne", "sum"),
    marge     = ("marge_ligne",   "sum"),
    nb_lignes = ("ligne_id",      "count"),
    remises   = ("remise_valeur", "sum")
).assign(
    taux_marge  = lambda x: (x["marge"] / x["ca"] * 100).round(1),
    panier_moy  = lambda x: (x["ca"] / x["nb_lignes"]).round(0)
).sort_values("marge", ascending=False)
print(renta.round(0))

# ── 2.2 CA et marge perdus (annulées + retournées) ────────────────────────
print("\n[2.2] CA et marge perdus par catégorie")
perdu = df_perdus.groupby("categorie").agg(
    ca_perdu    = ("montant_ligne", "sum"),
    marge_perdue= ("marge_ligne",   "sum")
).round(0)
print(perdu)

# ── 2.3 Rentabilité nette ─────────────────────────────────────────────────
print("\n[2.3] Rentabilité NETTE par catégorie")
renta_nette = renta.join(perdu)
renta_nette["ca_net"]         = renta_nette["ca"]    - renta_nette["ca_perdu"].fillna(0)
renta_nette["marge_nette"]    = renta_nette["marge"] - renta_nette["marge_perdue"].fillna(0)
renta_nette["taux_marge_net"] = (renta_nette["marge_nette"] / renta_nette["ca_net"] * 100).round(1)
print(renta_nette[["ca","ca_perdu","ca_net","marge","marge_nette","taux_marge_net"]].round(0))
# INSIGHT : La Maison a le meilleur taux de marge nette (54%) malgré un CA inférieur
# à l'Électronique. L'Électronique domine le CA brut mais ses pertes sont massives.

# ── 2.4 Top produits rentables ────────────────────────────────────────────
print("\n[2.4] Top 10 produits les plus rentables (livrés)")
top_prod = (
    df_livres.groupby(["nom_produit","categorie"]).agg(
        ca    = ("montant_ligne", "sum"),
        marge = ("marge_ligne",   "sum"),
        qte   = ("quantite",      "sum")
    ).assign(taux_marge=lambda x: (x["marge"] / x["ca"] * 100).round(1))
    .sort_values("marge", ascending=False)
    .head(10)
)
print(top_prod.round(0))

# ── 2.5 Produits à haute marge sous-exploités ─────────────────────────────
print("\n[2.5] Opportunités — produits à marge > 55% et CA sous la médiane")
prod_stats = (
    df_livres.groupby(["nom_produit","categorie"]).agg(
        ca    = ("montant_ligne", "sum"),
        marge = ("marge_ligne",   "sum"),
        qte   = ("quantite",      "sum")
    ).assign(taux_marge=lambda x: (x["marge"] / x["ca"] * 100).round(1))
)
seuil = prod_stats["ca"].median()
opport = prod_stats[(prod_stats["taux_marge"] > 55) & (prod_stats["ca"] < seuil)]
print(f"Seuil CA médian : {seuil:,.0f} €")
print(opport.sort_values("taux_marge", ascending=False).head(10).round(0))
# Ces produits sont très rentables mais peu visibles. Un meilleur référencement
# ou une mise en avant dans les recommandations pourrait booster la marge globale.

# ── 2.6 Visualisation rentabilité ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Rentabilité réelle par catégorie", fontsize=13, fontweight="bold")
cat_colors = {"Électronique":"#378ADD","Maison":"#1D9E75","Mode":"#7F77DD",
              "Sport":"#EF9F27","Beauté":"#D4537E"}

ax = axes[0]
cats = renta_nette.index.tolist()
x = range(len(cats))
w = 0.35
ax.bar([i - w/2 for i in x], renta_nette["ca"] / 1e3,       w, label="CA livré",   color=[cat_colors[c] for c in cats], alpha=0.9)
ax.bar([i + w/2 for i in x], renta_nette["ca_perdu"] / 1e3, w, label="CA perdu",   color="red", alpha=0.5)
ax.set_xticks(list(x)); ax.set_xticklabels(cats, rotation=20)
ax.set_title("CA livré vs CA perdu (k€)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%dk€"))
ax.legend()

ax = axes[1]
taux = renta_nette["taux_marge_net"].sort_values(ascending=True)
ax.barh(taux.index, taux.values, color=[cat_colors[c] for c in taux.index])
ax.set_title("Taux de marge nette par catégorie (%)")
ax.set_xlabel("%")
for i, v in enumerate(taux.values):
    ax.text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=9)

ax = axes[2]
marge_nette = renta_nette["marge_nette"].sort_values(ascending=True)
ax.barh(marge_nette.index, marge_nette.values / 1e3, color=[cat_colors[c] for c in marge_nette.index])
ax.set_title("Marge nette absolue par catégorie (k€)")
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%dk€"))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_rentabilite_categorie.png", dpi=130, bbox_inches="tight")
plt.show()
print("\n  ✓ Graphique sauvegardé : outputs/02_rentabilite_categorie.png")

---
## Section 8 — PERFORMANCE DES CODES PROMO


In [ ]:
# On évalue chaque code promo sur 3 critères :
#   1. Taux de friction (annulation + retours) qu'il génère
#   2. Montant des remises réellement accordées
#   3. CA net estimé = CA brut × (1 - taux_friction)
#
# On croise aussi avec les segments clients pour voir
# si les promos sont utilisées par les bons profils.

print("\n" + "=" * 65)
print("SECTION 3 — PERFORMANCE DES CODES PROMO")
print("=" * 65)

print(f"\nCommandes avec promo  : {commandes['a_promo'].sum()} ({commandes['a_promo'].mean()*100:.1f}%)")
print(f"Commandes sans promo  : {(~commandes['a_promo']).sum()}")
print(f"\nCodes utilisés :")
print(commandes["code_promo"].value_counts())

# ── 3.1 Détail par code promo ─────────────────────────────────────────────
print("\n[3.1] Performance par code promo")
promo_detail = (
    commandes[commandes["a_promo"]]
    .groupby("code_promo")
    .agg(
        nb_cmd      = ("commande_id", "count"),
        ca_brut     = ("montant_ttc", "sum"),
        panier_moy  = ("montant_ttc", "mean"),
        taux_annul  = ("statut", lambda x: round((x=="Annulée").sum()  / len(x) * 100, 1)),
        taux_retour = ("statut", lambda x: round((x=="Retournée").sum()/ len(x) * 100, 1)),
    )
    .assign(taux_friction=lambda x: (x["taux_annul"] + x["taux_retour"]).round(1))
)
print(promo_detail.round(1))

# ── 3.2 Remises accordées par code ────────────────────────────────────────
print("\n[3.2] Remises accordées par code promo")
remises_code = (
    df_full[df_full["a_promo"]]
    .merge(commandes[["commande_id","code_promo"]], on="commande_id")
    .groupby("code_promo")["remise_valeur"]
    .sum()
    .round(0)
)
print(remises_code)

# ── 3.3 Rendement net ─────────────────────────────────────────────────────
print("\n[3.3] Rendement net par code promo")
promo_detail["remise_accordee"] = remises_code
promo_detail["ca_net"]          = (promo_detail["ca_brut"] * (1 - promo_detail["taux_friction"]/100)).round(0)
promo_detail["cout_total"]      = promo_detail["remise_accordee"]
print(promo_detail[["nb_cmd","ca_brut","remise_accordee","ca_net","taux_friction"]].round(0))
# INSIGHT : SOLDES20 a le meilleur rendement net (17.3% friction).
# BIENVENUE10 est le pire (24.8%) — contre-productif pour un code d'acquisition.

# ── 3.4 Comparaison globale promo vs sans promo ───────────────────────────
print("\n[3.4] Comparaison globale promo vs sans promo")
comp = commandes.groupby("a_promo").agg(
    nb          = ("commande_id", "count"),
    ca_brut     = ("montant_ttc", "sum"),
    panier_moy  = ("montant_ttc", "mean"),
    panier_med  = ("montant_ttc", "median"),
    taux_annul  = ("statut", lambda x: round((x=="Annulée").sum()  / len(x) * 100, 1)),
    taux_retour = ("statut", lambda x: round((x=="Retournée").sum()/ len(x) * 100, 1)),
)
comp.index = ["Sans promo", "Avec promo"]
print(comp.round(1))
# INSIGHT CRITIQUE : +2.4 pts d'annulation avec promo (12% vs 8.8%)
# Le panier moyen est quasi identique — les promos ne font pas acheter plus.

# ── 3.5 Promo par segment client ─────────────────────────────────────────
print("\n[3.5] Impact des promos par segment client")
df_seg = commandes.merge(clients[["client_id","segment"]], on="client_id")
seg_promo = (
    df_seg.groupby(["segment","a_promo"])
    .agg(
        nb         = ("commande_id", "count"),
        ca         = ("montant_ttc", "sum"),
        panier     = ("montant_ttc", "mean"),
        taux_annul = ("statut", lambda x: round((x=="Annulée").sum()/len(x)*100, 1)),
    )
    .round(1)
)
seg_promo.index = pd.MultiIndex.from_tuples(
    [(s, "Avec promo" if p else "Sans promo") for s, p in seg_promo.index]
)
print(seg_promo)
# INSIGHT MAJEUR : Les clients Fidèles avec promo = 13.2% d'annulation vs 5.5% sans.
# On offre des remises à des clients qui achèteraient quand même, et ils annulent plus.

# ── 3.6 Remises sur commandes perdues ─────────────────────────────────────
print("\n[3.6] Analyse des remises accordées")
print(f"Remises totales accordées           : {df_full['remise_valeur'].sum():>10,.0f} €")
print(f"Dont sur commandes perdues (gaspillées) : {df_perdus['remise_valeur'].sum():>10,.0f} €")
print(f"Remises 'utiles' sur livrées        : {df_livres['remise_valeur'].sum():>10,.0f} €")
gaspillage = df_perdus["remise_valeur"].sum() / df_full["remise_valeur"].sum() * 100
print(f"\n  → {gaspillage:.1f}% des remises accordées sont associées à des commandes perdues")

# ── 3.7 Visualisation codes promo ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Performance des codes promo", fontsize=13, fontweight="bold")

# Taux de friction par code
ax = axes[0]
codes = promo_detail.sort_values("taux_friction")
colors_friction = ["#2ECC71" if t < 20 else "#F39C12" if t < 23 else "#E74C3C"
                   for t in codes["taux_friction"]]
ax.barh(codes.index, codes["taux_friction"], color=colors_friction)
ax.set_title("Taux de friction par code (%)")
ax.set_xlabel("Taux annul. + retour (%)")
for i, v in enumerate(codes["taux_friction"]):
    ax.text(v + 0.2, i, f"{v}%", va="center", fontsize=9)

# CA brut vs CA net
ax = axes[1]
x = range(len(promo_detail))
w = 0.35
ax.bar([i - w/2 for i in x], promo_detail["ca_brut"] / 1e3,  w, label="CA brut",  color="#3498DB", alpha=0.85)
ax.bar([i + w/2 for i in x], promo_detail["ca_net"]  / 1e3,  w, label="CA net",   color="#2ECC71", alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(promo_detail.index, rotation=15)
ax.set_title("CA brut vs CA net par code (k€)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%dk€"))
ax.legend()

# Taux d'annulation promo vs sans promo par segment
ax = axes[2]
seg_annul = df_seg.groupby(["segment","a_promo"])["statut"].apply(
    lambda x: (x=="Annulée").sum()/len(x)*100
).unstack()
seg_annul.columns = ["Sans promo", "Avec promo"]
seg_annul.plot(kind="bar", ax=ax, color=["#3498DB","#E74C3C"], alpha=0.85)
ax.set_title("Taux annulation par segment\n(promo vs sans promo)")
ax.set_xlabel("")
ax.set_ylabel("Taux d'annulation (%)")
ax.tick_params(axis="x", rotation=30)
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_codes_promo.png", dpi=130, bbox_inches="tight")
plt.show()
print("\n  ✓ Graphique sauvegardé : outputs/02_codes_promo.png")

---
## Section 9 — RÉSUMÉ DES INSIGHTS


In [ ]:
print("\n" + "=" * 65)
print("RÉSUMÉ DES INSIGHTS — RFM + RENTABILITÉ + PROMOS")
print("=" * 65)

insights = [
    # RFM
    ("RFM — Champions",
     f"76 clients génèrent 746 654 € (30% du CA actif). "
     "Récence ~45j, 6 commandes en moyenne. À chouchouter absolument."),
    ("RFM — Churn imminent",
     f"56 clients à risque, dernière commande il y a 285 jours en moyenne. "
     f"CA historique : 412 917 €. Campagne de réactivation urgente."),
    ("RFM — Inactifs",
     "118 clients (24% de la base) inactifs depuis +1 an. "
     "Tentative de réactivation 'dernière chance' puis nettoyage de base."),
    # Rentabilité
    ("Rentabilité — Maison = vraie star",
     "53% de marge brute, 54% nette. Plus rentable que l'Électronique malgré un CA inférieur. "
     "Produits à mettre en avant en priorité."),
    ("Rentabilité — Électronique surévalué",
     "Domine le CA brut mais 46% de marge nette seulement. "
     f"269 523 € de CA perdu sur retours/annulations."),
    ("Rentabilité — Opportunités",
     "10 produits avec taux de marge > 55% et CA sous la médiane. "
     "Mieux les référencer = gain de marge directement actionnable."),
    # Promos
    ("Promo — BIENVENUE10 contre-productif",
     "24.8% de friction — le pire de tous. "
     "Le code censé convertir les nouveaux génère le plus d'annulations."),
    ("Promo — SOLDES20 = seul code rentable",
     "17.3% de friction, le seul sous la moyenne. "
     "Amplifier une intention d'achat forte > créer une intention artificielle."),
    ("Promo — Clients fidèles",
     "Taux annulation 5.5% sans promo → 13.2% avec promo. "
     "Les codes promo dégradent le comportement des meilleurs clients."),
]

for titre, detail in insights:
    print(f"\n  ► {titre}")
    print(f"    {detail}")

print("\n" + "=" * 65)
print("FIN DU NOTEBOOK 02")
print("=" * 65)

In [ ]:
# =============================================================================
# PROJET E-COMMERCE — ANALYSE COMPLÈTE
# Notebook 03 : Saisonnalité · Géographie · Stocks · Modèles Prédictifs
# =============================================================================

                              mean_absolute_error, ConfusionMatrixDisplay)


# DATA_DIR et OUTPUT_DIR définis dans la cellule de configuration

commandes  = pd.read_csv(DATA_DIR / "data_commandes.csv",        sep=";")
lignes     = pd.read_csv(DATA_DIR / "data_lignes_commandes.csv", sep=";")
clients    = pd.read_csv(DATA_DIR / "data_clients.csv",          sep=";")
produits   = pd.read_csv(DATA_DIR / "data_produits.csv",         sep=";")
calendrier = pd.read_csv(DATA_DIR / "dim_calendrier.csv",        sep=";")

# Pré-traitements communs
commandes["date_commande"]   = pd.to_datetime(commandes["date_commande"])
clients["date_inscription"]  = pd.to_datetime(clients["date_inscription"], dayfirst=True)
calendrier["date_complete"]  = pd.to_datetime(calendrier["date_complete"])
commandes["annee"]           = commandes["date_commande"].dt.year
commandes["mois_num"]        = commandes["date_commande"].dt.month
commandes["mois_seq"]        = (commandes["annee"] - 2023) * 12 + commandes["mois_num"]
commandes["jour_semaine"]    = commandes["date_commande"].dt.day_name()
commandes["heure"]           = commandes["heure_commande"].str[:2].astype(int)
commandes["trimestre"]       = commandes["date_commande"].dt.quarter
commandes["a_promo"]         = commandes["code_promo"].notna()
produits["taux_marge"]       = ((produits["prix_unitaire"] - produits["cout_achat"])
                                 / produits["prix_unitaire"] * 100).round(2)

ORDRE_JOURS = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
JOURS_FR    = {"Monday":"Lundi","Tuesday":"Mardi","Wednesday":"Mercredi",
               "Thursday":"Jeudi","Friday":"Vendredi","Saturday":"Samedi","Sunday":"Dimanche"}

df_clients = commandes.merge(clients[["client_id","segment","region","canal_acquisition"]], on="client_id")

---
## Section 10 — SAISONNALITÉ & TENDANCES TEMPORELLES


In [ ]:
print("=" * 65)
print("SECTION 1 — SAISONNALITÉ & TENDANCES TEMPORELLES")
print("=" * 65)

# ── 1.1 CA mensuel par année ──────────────────────────────────────────────
print("\n[1.1] CA mensuel par année")
ca_mois = (commandes.groupby(["annee","mois_num"])
           .agg(nb_cmd=("commande_id","count"),
                ca=("montant_ttc","sum"),
                panier=("montant_ttc","mean"))
           .round(0))
print(ca_mois)

# ── 1.2 Évolution 2023 → 2024 (mois communs) ────────────────────────────
print("\n[1.2] Évolution mensuelle 2023 → 2024")
c2023 = commandes[commandes["annee"] == 2023]
c2024 = commandes[commandes["annee"] == 2024]
evols = []
for m in range(1, 11):
    ca23 = c2023[c2023["mois_num"]==m]["montant_ttc"].sum()
    ca24 = c2024[c2024["mois_num"]==m]["montant_ttc"].sum()
    evol = (ca24 - ca23) / ca23 * 100 if ca23 > 0 else 0
    evols.append({"mois": m, "ca_2023": ca23, "ca_2024": ca24, "evolution_pct": round(evol,1)})
    print(f"  Mois {m:02d}: 2023={ca23:>9,.0f}€  2024={ca24:>9,.0f}€  évol={evol:+.1f}%")
evols_df = pd.DataFrame(evols)
# INSIGHT : Pas de tendance claire sur 2 ans. Fort rebond en Mar/Jun/Sep 2024,
# mais Jan/Mai/Oct 2024 reculent vs 2023. Volatilité mensuelle élevée.

# ── 1.3 CA par jour de la semaine ─────────────────────────────────────────
print("\n[1.3] CA par jour de la semaine")
ca_jour = (commandes.groupby("jour_semaine")
           .agg(nb=("commande_id","count"),
                ca=("montant_ttc","sum"),
                panier=("montant_ttc","mean"))
           .reindex(ORDRE_JOURS))
ca_jour.index = [JOURS_FR[j] for j in ca_jour.index]
print(ca_jour.round(0))
# INSIGHT : Weekend (Sam+Dim) = panier 10% plus élevé que semaine.
# Dimanche = meilleur jour global (477 674 €).

# ── 1.4 CA par tranche horaire ───────────────────────────────────────────
print("\n[1.4] CA par tranche horaire")
bins   = [6, 9, 12, 15, 18, 21, 24]
labels = ["6-9h","9-12h","12-15h","15-18h","18-21h","21-24h"]
commandes["tranche"] = pd.cut(commandes["heure"], bins=bins, labels=labels, right=False)
print(commandes.groupby("tranche").agg(nb=("commande_id","count"),
                                        ca=("montant_ttc","sum")).round(0))
# INSIGHT : Double pic 12-15h et 18-21h → créneau optimal pour envoi d'emails/notifs.

# ── 1.5 Impact jours fériés et weekends ──────────────────────────────────
print("\n[1.5] Impact jours fériés")
cal_m = calendrier[["date_complete","est_ferie","est_weekend"]].copy()
cal_m["date_only"] = cal_m["date_complete"].dt.date
commandes["date_only"] = commandes["date_commande"].dt.date
cmd_cal = commandes.merge(cal_m, on="date_only", how="left")
print("Jours fériés vs jours normaux :")
print(cmd_cal.groupby("est_ferie").agg(nb=("commande_id","count"),
                                        ca=("montant_ttc","sum"),
                                        panier=("montant_ttc","mean")).round(0))
print("\nWeekend vs semaine :")
print(cmd_cal.groupby("est_weekend").agg(nb=("commande_id","count"),
                                          ca=("montant_ttc","sum"),
                                          panier=("montant_ttc","mean")).round(0))

# ── 1.6 CA par trimestre ─────────────────────────────────────────────────
print("\n[1.6] CA par trimestre")
print(commandes.groupby(["annee","trimestre"])["montant_ttc"]
      .agg(["sum","count","mean"]).round(0))

# ── 1.7 Visualisation saisonnalité ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Saisonnalité & Tendances temporelles", fontsize=13, fontweight="bold")

# CA mensuel 2023 vs 2024
ax = axes[0]
x = evols_df["mois"]
ax.plot(x, evols_df["ca_2023"]/1e3, marker="o", label="2023", color="#378ADD", linewidth=2)
ax.plot(x, evols_df["ca_2024"]/1e3, marker="s", label="2024", color="#1D9E75", linewidth=2)
ax.fill_between(x, evols_df["ca_2023"]/1e3, evols_df["ca_2024"]/1e3,
                alpha=0.1, color="gray")
ax.set_title("CA mensuel 2023 vs 2024 (k€)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%dk€"))
ax.set_xlabel("Mois")
ax.legend()

# CA par jour de la semaine
ax = axes[1]
colors_jour = ["#7F77DD" if j in ["Samedi","Dimanche"] else "#378ADD" for j in ca_jour.index]
ax.bar(ca_jour.index, ca_jour["ca"]/1e3, color=colors_jour, alpha=0.85)
ax.set_title("CA par jour de la semaine (k€)")
ax.set_xlabel("")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%dk€"))
ax.tick_params(axis="x", rotation=30)

# CA par tranche horaire
ax = axes[2]
tr_data = commandes.groupby("tranche")["montant_ttc"].sum()
colors_tr = ["#F39C12" if t in ["12-15h","18-21h"] else "#3498DB" for t in tr_data.index]
ax.bar(tr_data.index, tr_data.values/1e3, color=colors_tr, alpha=0.85)
ax.set_title("CA par tranche horaire (k€)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%dk€"))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_saisonnalite.png", dpi=130, bbox_inches="tight")
plt.show()
print("\n  ✓ outputs/03_saisonnalite.png")

---
## Section 11 — ANALYSE GÉOGRAPHIQUE


In [ ]:
print("\n" + "=" * 65)
print("SECTION 2 — ANALYSE GÉOGRAPHIQUE")
print("=" * 65)

# ── 2.1 Performance par région ───────────────────────────────────────────
print("\n[2.1] Performance complète par région")
reg = df_clients.groupby("region").apply(lambda x: pd.Series({
    "nb_clients":      x["client_id"].nunique(),
    "nb_cmd":          len(x),
    "ca":              x["montant_ttc"].sum(),
    "panier_moy":      x["montant_ttc"].mean(),
    "ca_par_client":   x["montant_ttc"].sum() / x["client_id"].nunique(),
    "cmd_par_client":  len(x) / x["client_id"].nunique(),
    "taux_annul":      round((x["statut"]=="Annulée").sum() / len(x) * 100, 1),
    "taux_retour":     round((x["statut"]=="Retournée").sum() / len(x) * 100, 1),
    "taux_friction":   round(x["statut"].isin(["Annulée","Retournée"]).sum() / len(x) * 100, 1),
})).sort_values("ca", ascending=False)
print(reg.round(1))
# INSIGHT CLÉ : Normandie = 12.7% friction (meilleure région), IDF = 25% (pire).
# Bourgogne-FC = meilleur CA/client (6 925 €) malgré seulement 24 clients.

# ── 2.2 Segments par région ───────────────────────────────────────────────
print("\n[2.2] Distribution des segments par région")
seg_reg = df_clients.groupby(["region","segment"]).size().unstack(fill_value=0)
print(seg_reg)

# ── 2.3 Panier moyen croisé région × segment ─────────────────────────────
print("\n[2.3] Panier moyen par région × segment")
print(df_clients.groupby(["region","segment"])["montant_ttc"].mean().round(0).unstack())

# ── 2.4 Canal d'acquisition dominant par région ──────────────────────────
print("\n[2.4] Canal d'acquisition dominant par région")
canal_dom = (df_clients.groupby(["region","canal_acquisition"])["montant_ttc"]
             .sum().unstack(fill_value=0))
canal_dom["dominant"] = canal_dom.idxmax(axis=1)
print(canal_dom[["dominant"]])

# ── 2.5 Visualisation géo ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Performance géographique", fontsize=13, fontweight="bold")

ax = axes[0]
reg_plot = reg.sort_values("ca")
colors_reg = ["#E74C3C" if r=="Île-de-France"
              else "#2ECC71" if r=="Normandie"
              else "#3498DB" for r in reg_plot.index]
ax.barh(reg_plot.index, reg_plot["ca"]/1e3, color=colors_reg, alpha=0.85)
ax.set_title("CA total par région (k€)")
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%dk€"))

ax = axes[1]
reg_fric = reg.sort_values("taux_friction")
colors_fric = ["#2ECC71" if t < 15
               else "#F39C12" if t < 22
               else "#E74C3C" for t in reg_fric["taux_friction"]]
bars = ax.barh(reg_fric.index, reg_fric["taux_friction"], color=colors_fric, alpha=0.85)
ax.set_title("Taux de friction par région (%)")
ax.set_xlabel("%")
for i, v in enumerate(reg_fric["taux_friction"]):
    ax.text(v + 0.2, i, f"{v:.1f}%", va="center", fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_geo.png", dpi=130, bbox_inches="tight")
plt.show()
print("\n  ✓ outputs/03_geo.png")

---
## Section 12 — ANALYSE DES STOCKS


In [ ]:
print("\n" + "=" * 65)
print("SECTION 3 — ANALYSE DES STOCKS")
print("=" * 65)

# Quantités vendues par produit (commandes livrées uniquement)
cmd_livrees  = commandes[commandes["statut"] == "Livrée"]["commande_id"]
qte_vendue   = (lignes[lignes["commande_id"].isin(cmd_livrees)]
                .groupby("produit_id")["quantite"].sum().reset_index()
                .rename(columns={"quantite": "qte_vendue"}))

stocks = produits.merge(qte_vendue, on="produit_id", how="left")
stocks["qte_vendue"]   = stocks["qte_vendue"].fillna(0)
# Rotation = nb fois que le stock a été renouvelé théoriquement
stocks["rotation"]     = (stocks["qte_vendue"] / stocks["stock_actuel"]).round(2)
# Jours de stock restants = stock / (ventes annualisées par jour)
stocks["jours_stock"]  = (
    stocks["stock_actuel"] / (stocks["qte_vendue"] / 365)
).replace([np.inf, np.nan], 9999).round(0)
stocks["en_alerte"]    = stocks["stock_actuel"] <= stocks["stock_min"]
stocks["sur_stock"]    = stocks["stock_actuel"] > stocks["stock_min"] * 3
stocks["valeur_stock"] = stocks["stock_actuel"] * stocks["cout_achat"]

# ── 3.1 Alertes stock ────────────────────────────────────────────────────
print("\n[3.1] Produits en alerte stock critique")
print(stocks[stocks["en_alerte"]][["nom_produit","categorie","stock_actuel",
                                    "stock_min","qte_vendue","rotation"]].sort_values("stock_actuel"))

# ── 3.2 Produits en sur-stock ─────────────────────────────────────────────
print(f"\n[3.2] Produits en sur-stock : {stocks['sur_stock'].sum()} / {len(stocks)}")
print("\nTop 10 sur-stocks les + extrêmes (par jours de stock restants) :")
print(stocks[stocks["sur_stock"]].sort_values("jours_stock", ascending=False)
      .head(10)[["nom_produit","categorie","stock_actuel","qte_vendue","jours_stock"]].to_string())
# NOTE : Sérum vitamine C = 9.5 ans de stock ! Capital mort immobilisé.

# ── 3.3 Rotation par catégorie ────────────────────────────────────────────
print("\n[3.3] Analyse stocks par catégorie")
print(stocks.groupby("categorie").agg(
    stock_total      = ("stock_actuel",  "sum"),
    valeur_stock     = ("valeur_stock",  "sum"),
    qte_vendue_tot   = ("qte_vendue",    "sum"),
    rotation_moy     = ("rotation",      "mean"),
    nb_alertes       = ("en_alerte",     "sum"),
    nb_surstocks     = ("sur_stock",     "sum"),
    jours_stock_moy  = ("jours_stock",   lambda x: x[x < 9999].mean())
).round(1))

# ── 3.4 Valeur totale du stock ────────────────────────────────────────────
print(f"\n[3.4] Valeur totale du stock (au coût d'achat) :")
print(f"  Total           : {stocks['valeur_stock'].sum():>12,.0f} €")
print(stocks.groupby("categorie")["valeur_stock"].sum().sort_values(ascending=False).round(0))
# INSIGHT : Électronique immobilise 1.82M€ de stock = 63% de la valeur totale.
# Optimiser les niveaux de stock Électronique libère du cash significatif.

# ── 3.5 Visualisation stocks ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Analyse des stocks", fontsize=13, fontweight="bold")

ax = axes[0]
alerte_cat = stocks.groupby("categorie")["en_alerte"].sum()
surstock_cat = stocks.groupby("categorie")["sur_stock"].sum()
x = range(len(alerte_cat))
w = 0.35
ax.bar([i-w/2 for i in x], alerte_cat.values, w, label="Alertes", color="#E74C3C", alpha=0.85)
ax.bar([i+w/2 for i in x], surstock_cat.values, w, label="Sur-stocks", color="#F39C12", alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(alerte_cat.index, rotation=20)
ax.set_title("Alertes vs Sur-stocks par catégorie")
ax.legend()

ax = axes[1]
rot_top = stocks.sort_values("rotation", ascending=False).head(10)
colors_rot = ["#E74C3C" if r > 5 else "#F39C12" if r > 2 else "#3498DB"
              for r in rot_top["rotation"]]
ax.barh(rot_top["nom_produit"], rot_top["rotation"], color=colors_rot, alpha=0.85)
ax.set_title("Top 10 rotations de stock")
ax.set_xlabel("Rotation (qté vendue / stock)")

ax = axes[2]
val_cat = stocks.groupby("categorie")["valeur_stock"].sum() / 1e3
val_cat.plot(kind="pie", ax=ax, autopct="%1.1f%%",
             colors=["#378ADD","#1D9E75","#7F77DD","#EF9F27","#D4537E"])
ax.set_title("Valeur du stock par catégorie (k€)")
ax.set_ylabel("")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_stocks.png", dpi=130, bbox_inches="tight")
plt.show()
print("\n  ✓ outputs/03_stocks.png")

---
## Section 13 — MODÈLES PRÉDICTIFS


In [ ]:
print("\n" + "=" * 65)
print("SECTION 4 — MODÈLES PRÉDICTIFS")
print("=" * 65)

date_ref = commandes["date_commande"].max()

# ── Préparation dataset client enrichi ───────────────────────────────────
cmd_valides = commandes[~commandes["statut"].isin(["Annulée","Retournée"])]

rfm = cmd_valides.groupby("client_id").agg(
    recence   = ("date_commande", lambda x: (date_ref - x.max()).days),
    frequence = ("commande_id",   "count"),
    valeur    = ("montant_ttc",   "sum"),
    panier_moy= ("montant_ttc",   "mean"),
).reset_index()

annul_cli = commandes.groupby("client_id").apply(lambda x: pd.Series({
    "nb_annulations":   (x["statut"]=="Annulée").sum(),
    "nb_retours":       (x["statut"]=="Retournée").sum(),
    "nb_cmd_total":     len(x),
    "a_utilise_promo":  x["code_promo"].notna().any().astype(int),
})).reset_index()

rfm = rfm.merge(annul_cli, on="client_id")
rfm = rfm.merge(clients[["client_id","segment","region",
                           "canal_acquisition","date_inscription"]], on="client_id")
rfm["taux_annul"]  = rfm["nb_annulations"] / rfm["nb_cmd_total"]
rfm["taux_retour"] = rfm["nb_retours"]     / rfm["nb_cmd_total"]
rfm["anciennete"]  = (date_ref - rfm["date_inscription"]).dt.days

le = LabelEncoder()
rfm["segment_enc"] = le.fit_transform(rfm["segment"])
rfm["canal_enc"]   = le.fit_transform(rfm["canal_acquisition"])
rfm["region_enc"]  = le.fit_transform(rfm["region"])


# ─── MODÈLE 1 : PRÉDICTION DU CHURN ──────────────────────────────────────
# Définition : client churn = dernière commande > 180 jours
# Objectif   : identifier les clients qui risquent de partir
#              AVANT qu'ils partent → campagne de rétention proactive
print("\n[4.1] MODÈLE 1 — Prédiction du churn")
rfm["churn"] = (rfm["recence"] > 180).astype(int)
print(f"Taux de churn actuel : {rfm['churn'].mean()*100:.1f}%")
print(f"Distribution : {rfm['churn'].value_counts().to_dict()}")

FEATURES_CHURN = ["frequence","valeur","panier_moy","anciennete","taux_annul",
                  "taux_retour","nb_cmd_total","a_utilise_promo","segment_enc","canal_enc"]
X = rfm[FEATURES_CHURN]
y = rfm["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_churn = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=5,
                                   random_state=42, class_weight="balanced")
rf_churn.fit(X_train, y_train)
y_pred  = rf_churn.predict(X_test)
y_proba = rf_churn.predict_proba(X_test)[:, 1]

print("\nRapport de classification :")
print(classification_report(y_test, y_pred))
auc = roc_auc_score(y_test, y_proba)
print(f"AUC-ROC : {auc:.3f}")

cv_scores = cross_val_score(rf_churn, X, y, cv=5, scoring="roc_auc")
print(f"Cross-validation AUC (5-fold) : {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Importance des features
feat_imp_churn = pd.Series(rf_churn.feature_importances_,
                            index=FEATURES_CHURN).sort_values(ascending=False)
print("\nImportance des variables :")
print(feat_imp_churn.round(3))
# LECTURE : Fréquence (20.5%) et Valeur (20.1%) sont les meilleurs prédicteurs.
# Le segment déclaré (5%) est bien moins informatif que le comportement réel.

# Score de churn pour tous les clients
rfm["proba_churn"] = rf_churn.predict_proba(X)[:, 1]
print("\nTop 10 clients à risque élevé (non encore churnés) :")
top_churn = (rfm[rfm["churn"] == 0]
             .sort_values("proba_churn", ascending=False)
             .head(10))
print(top_churn[["client_id","segment","recence","frequence","valeur","proba_churn"]].round(3))


# ─── MODÈLE 2 : PRÉVISION DU CA MENSUEL ──────────────────────────────────
# Méthode : Gradient Boosting sur features temporelles + lags
# Lags : CA du mois précédent (lag1), du mois d'avant (lag2), moyenne 3 mois (ma3)
# Variables cycliques : sin/cos du mois pour capturer la saisonnalité
print("\n[4.2] MODÈLE 2 — Prévision du CA mensuel")

ca_mois_ts = (commandes.groupby("mois_seq").agg(
    ca       = ("montant_ttc", "sum"),
    nb_cmd   = ("commande_id", "count"),
    panier   = ("montant_ttc", "mean"),
).reset_index())

ca_mois_ts["mois_num"] = ((ca_mois_ts["mois_seq"] - 1) % 12) + 1
ca_mois_ts["sin_mois"] = np.sin(2 * np.pi * ca_mois_ts["mois_num"] / 12)
ca_mois_ts["cos_mois"] = np.cos(2 * np.pi * ca_mois_ts["mois_num"] / 12)
ca_mois_ts["ca_lag1"]  = ca_mois_ts["ca"].shift(1)
ca_mois_ts["ca_lag2"]  = ca_mois_ts["ca"].shift(2)
ca_mois_ts["ca_ma3"]   = ca_mois_ts["ca"].rolling(3).mean()
ca_mois_ts = ca_mois_ts.dropna()

FEATURES_TS = ["mois_seq","mois_num","sin_mois","cos_mois",
               "ca_lag1","ca_lag2","ca_ma3","nb_cmd","panier"]
X_ts = ca_mois_ts[FEATURES_TS]
y_ts = ca_mois_ts["ca"]

# Train sur tout sauf les 3 derniers mois (test)
X_train_ts, X_test_ts = X_ts.iloc[:-3], X_ts.iloc[-3:]
y_train_ts, y_test_ts = y_ts.iloc[:-3], y_ts.iloc[-3:]

gb = GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                learning_rate=0.05, random_state=42)
gb.fit(X_train_ts, y_train_ts)
y_pred_ts = gb.predict(X_test_ts)

mae  = mean_absolute_error(y_test_ts, y_pred_ts)
mape = np.mean(np.abs((y_test_ts.values - y_pred_ts) / y_test_ts.values)) * 100
print(f"MAE  : {mae:,.0f} €   MAPE : {mape:.1f}%")
print("\nDétail test :")
for i, (real, pred) in enumerate(zip(y_test_ts, y_pred_ts)):
    print(f"  Mois test {i+1}: Réel={real:>9,.0f}€  Prédit={pred:>9,.0f}€  Écart={abs(real-pred)/real*100:.1f}%")

# Prévisions futures (Nov, Déc 2024)
ca_history = list(y_ts.values)
for m_offset in [1, 2]:
    seq = ca_mois_ts["mois_seq"].max() + m_offset
    mn  = ((seq - 1) % 12) + 1
    row = pd.DataFrame({
        "mois_seq":[seq], "mois_num":[mn],
        "sin_mois":[np.sin(2*np.pi*mn/12)],
        "cos_mois":[np.cos(2*np.pi*mn/12)],
        "ca_lag1":[ca_history[-1]], "ca_lag2":[ca_history[-2]],
        "ca_ma3":[np.mean(ca_history[-3:])],
        "nb_cmd":[ca_mois_ts["nb_cmd"].mean()],
        "panier":[ca_mois_ts["panier"].mean()],
    })
    pred = gb.predict(row)[0]
    ca_history.append(pred)
    print(f"  Prévision mois {mn}/2024 : {pred:,.0f} €")


# ─── MODÈLE 3 : SCORING PROPENSION À RACHETER ────────────────────────────
# Objectif : parmi les clients inactifs (90+ jours), qui a la plus forte
#            probabilité de racheter dans les 60 prochains jours ?
# Cible simulée : client ayant commandé dans les 60 derniers jours de l'historique
print("\n[4.3] MODÈLE 3 — Scoring propension à racheter")
rfm["va_racheter"] = (rfm["recence"] <= 60).astype(int)
print(f"Clients avec achat récent (<= 60j) : {rfm['va_racheter'].sum()}")

FEATURES_PROP = ["frequence","valeur","panier_moy","anciennete","taux_annul",
                 "nb_cmd_total","segment_enc","canal_enc"]
X3 = rfm[FEATURES_PROP]
y3 = rfm["va_racheter"]

rf_prop = RandomForestClassifier(n_estimators=200, max_depth=5,
                                  random_state=42, class_weight="balanced")
rf_prop.fit(X3, y3)
rfm["score_propension"] = rf_prop.predict_proba(X3)[:, 1]

print("\nTop 15 clients inactifs (>90j) à fort score de propension :")
inactifs_potentiels = (rfm[rfm["recence"] > 90]
                       .sort_values("score_propension", ascending=False)
                       .head(15))
print(inactifs_potentiels[["client_id","segment","recence","frequence",
                            "valeur","score_propension"]].round(3))

# ── 4.4 Visualisation modèles ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Modèles prédictifs — performances & résultats", fontsize=13, fontweight="bold")

# Importance features churn
ax = axes[0]
feat_imp_churn.sort_values().plot(kind="barh", ax=ax, color="#E74C3C", alpha=0.85)
ax.set_title("Importance des variables\n(modèle churn)")
ax.set_xlabel("Importance")

# CA réel vs prédit sur la série temporelle
ax = axes[1]
ax.plot(ca_mois_ts["mois_seq"], ca_mois_ts["ca"]/1e3,
        label="Réel", color="#2980B9", linewidth=1.5)
train_seq  = X_train_ts["mois_seq"]
train_pred = gb.predict(X_train_ts)
ax.plot(train_seq, train_pred/1e3, "--", label="Prédit (train)",
        color="#E67E22", linewidth=1.2, alpha=0.8)
test_seq  = X_test_ts["mois_seq"]
ax.scatter(test_seq, y_pred_ts/1e3, color="#E74C3C", s=60, zorder=5, label="Prédit (test)")
ax.set_title(f"CA réel vs prédit (k€)\nMAE={mae/1e3:.1f}k€  MAPE={mape:.1f}%")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%dk€"))
ax.legend(fontsize=8)

# Distribution des scores de churn
ax = axes[2]
ax.hist(rfm[rfm["churn"]==0]["proba_churn"], bins=15, alpha=0.7,
        color="#2ECC71", label="Non churné")
ax.hist(rfm[rfm["churn"]==1]["proba_churn"], bins=15, alpha=0.7,
        color="#E74C3C", label="Churné")
ax.axvline(x=0.5, color="black", linestyle="--", linewidth=1)
ax.set_title(f"Distribution scores de churn\nAUC={auc:.3f}")
ax.set_xlabel("Probabilité de churn")
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_predictions.png", dpi=130, bbox_inches="tight")
plt.show()
print("\n  ✓ outputs/03_predictions.png")

# Sauvegarder les scores pour le dashboard Streamlit
scores_export = rfm[["client_id","segment","region","recence","frequence",
                      "valeur","panier_moy","proba_churn","score_propension",
                      "churn","va_racheter"]].round(3)
scores_export.to_csv(OUTPUT_DIR / "scores_clients.csv", index=False)
print("  ✓ outputs/scores_clients.csv (scores ML par client)")

---
## Section 14 — RÉSUMÉ FINAL


In [ ]:
print("\n" + "=" * 65)
print("RÉSUMÉ — SAISONNALITÉ + GÉO + STOCKS + PRÉDICTIONS")
print("=" * 65)
insights = [
    ("Saisonnalité — meilleur créneau",
     "Dimanche + 12-15h ou 18-21h. Planifier emailings et notifs sur ces créneaux."),
    ("Saisonnalité — 2023 vs 2024",
     "Pas de tendance haussière claire. Volatilité mensuelle élevée (+33% en Jun, -21% en Mai 2024)."),
    ("Géo — Normandie = benchmark",
     "12.7% friction, meilleure région. Étudier et répliquer ses pratiques."),
    ("Géo — IDF = alarme",
     "25% friction, CA/client moyen malgré marché premium. Problème logistique ou offre inadaptée."),
    ("Stocks — ruptures imminentes",
     "Armoire (2 unités) et Bâtons marche (3 unités) : réapprovisionnement urgent."),
    ("Stocks — sur-stockage massif",
     "83/100 produits en sur-stock. Sérum vit. C = 9.5 ans de stock. 2.89M€ immobilisés."),
    ("ML — Churn AUC 0.712",
     "Modèle opérationnel. Fréquence et valeur > segment déclaré pour prédire le churn."),
    ("ML — CA MAE 9 232 €",
     "Erreur ~6% sur les prévisions mensuelles. Utilisable pour planning budgétaire."),
    ("ML — 39% de churn",
     "Priorité absolue sur la rétention. 187 clients perdus, 56 en danger immédiat."),
]
for titre, detail in insights:
    print(f"\n  ► {titre}")
    print(f"    {detail}")

print("\n" + "=" * 65)
print("FIN DU NOTEBOOK 03")
print("=" * 65)

---
## 🏁 Résumé des insights clés

| # | Insight | Impact |
|---|---|---|
| 1 | **623 925 €** de CA perdu (annul. + retours) | 20% du CA brut |
| 2 | **76 Champions** génèrent 30% du CA actif | Protéger ces clients |
| 3 | **Maison** = meilleure marge nette (54%) | Mettre en avant |
| 4 | **BIENVENUE10** = 24,8% friction | Revoir ce code promo |
| 5 | **39% de churn** — 187 clients perdus | Rétention urgente |
| 6 | **Armoire + Bâtons marche** en rupture imminente | Réappro immédiat |
| 7 | **IDF** = 25% friction sur marché stratégique | Investiguer |
| 8 | **DPD** = meilleur transporteur (17,4%) | Orienter les expéditions |
| 9 | **AUC 0.712** modèle churn opérationnel | Campagnes ciblées |
